# 1. NaiveRAG 시스템 구현

- 전체 문서 파일을 바탕으로 NaiveRAG를 구현하고 Langfuse를 사용하여 추적해봅니다.

## 환경설정

In [5]:
import os

# 1. macOS OpenMP 충돌 방지 (반드시 최상단 실행)
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

# 2. 토크나이저 병렬 처리 충돌 방지
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("환경 설정 완료. 이제 다음 셀들을 실행하세요.")

환경 설정 완료. 이제 다음 셀들을 실행하세요.


In [6]:
from dotenv import load_dotenv

load_dotenv()

True

## RAG 기본 파이프라인

In [8]:
# 라이브러리 임포트

from langchain_text_splitters import RecursiveCharacterTextSplitter     # 분할기
from langchain_community.document_loaders import PyMuPDFLoader          # 로더
from langchain_community.vectorstores import FAISS                      # 벡터스토어
from langchain_core.output_parsers import StrOutputParser              
from langchain_core.runnables import RunnablePassthrough                
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langfuse.langchain import CallbackHandler                          # llm 관측도구                         # llm 관측 ㄷ                                            # 경로


### 사전 준비단계

In [31]:
# 단계 1: 문서 로드
import os                       
import json
from pathlib import Path  

pdf_dir = '/Users/apple/Team2-RAG-Project/data/pdfs'
parsed_dir = '/Users/apple/Team2-RAG-Project/data/parsed'
os.makedirs(parsed_dir, exist_ok=True)

pdf_files = sorted([f for f in os.listdir(pdf_dir)])

for file_name in pdf_files:
    path = os.path.join(pdf_dir, file_name)
    loader = PyMuPDFLoader(path)
    docs = loader.load()

    out_path = os.path.join(parsed_dir, f"{os.path.splitext(file_name)[0]}.jsonl")
    with open(out_path, "w", encoding="utf-8") as f:
        for d in docs:
            record = {
                "source_file": file_name,          # 최소 메타데이터 1
                "page": d.metadata.get("page"),     # 최소 메타데이터 2
                "text": d.page_content
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict



- 대용량 파일 파싱, 후속 정제/디버깅에 용이하도록 data/parsed에 jsonl파일로 저장
- txt/jsonl/markdown/html 중. 
정보의 명확한 구분과 파일 자체에 내장된 메타데이터인 page 보존을 위해 jsonl파일을 선택

In [32]:
# 단계 1-2: 데이터 전처리

import json, re, unicodedata
from pathlib import Path
from collections import Counter

# [입력/출력 경로] data/parsed -> data/preprocessed
src_dir = Path("/Users/apple/Team2-RAG-Project/data/parsed")
dst_dir = Path("/Users/apple/Team2-RAG-Project/data/preprocessed")
dst_dir.mkdir(parents=True, exist_ok=True)

# 페이지 번호 헤더 패턴 예: "- 1 -"
page_no = re.compile(r"^-\s*\d+\s*-$")

# [입력] data/parsed의 각 jsonl 파일을 순회
for fp in src_dir.glob("*.jsonl"):
    rows = [json.loads(line) for line in fp.open("r", encoding="utf-8")]
    pages = [r.get("text", "") for r in rows]

    # [전처리-헤더/푸터 탐지] 페이지 상단/하단에서 반복되는 줄 찾기
    top, bottom = Counter(), Counter()
    for t in pages:
        ls = [x.strip() for x in t.splitlines() if x.strip()]
        top.update(ls[:2])       # 상단 2줄 후보
        bottom.update(ls[-2:])   # 하단 2줄 후보
    n = max(1, len(pages))
    repeated = {k for k, v in top.items() if v / n >= 0.5} | {k for k, v in bottom.items() if v / n >= 0.5}

    # [출력] 같은 파일명으로 data/preprocessed에 저장
    out_path = dst_dir / fp.name
    with out_path.open("w", encoding="utf-8") as out:
        for r in rows:
            s = r.get("text", "")

            # [전처리-제어문자 제거]
            s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C" or ch in "\n\t")

            # [전처리-헤더/푸터 제거] 반복줄 + 페이지번호 패턴 제거
            s = "\n".join(
                ln for ln in s.splitlines()
                if ln.strip() not in repeated and not page_no.match(ln.strip())
            )

            # [전처리-공백/줄바꿈 정리]
            s = re.sub(r"\r\n?", "\n", s)   # 줄바꿈 통일
            s = re.sub(r"[ \t]+", " ", s)   # 연속 공백/탭 -> 1칸
            s = re.sub(r"\n{2,}", "\n", s)  # 연속 빈 줄 축소
            s = s.strip()

            r["text"] = s
            out.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"완료: {dst_dir}")



완료: /Users/apple/Team2-RAG-Project/data/preprocessed


- 제어문자 제거: 보이지 않는 제어문자(\n, \t 제외) 삭제
- 헤더/푸터 제거: 페이지 상/하단에서 반복되는 줄 + - 1 - 같은 페이지 번호 줄 제거
- 공백/줄바꿈 정리: 연속 공백/탭 축소, 줄바꿈 형식 통일, 빈 줄 과다 제거

In [33]:
# 1-3) 메타데이터와 결합, Document 객체 생성
import re, unicodedata
import pandas as pd
from langchain_core.documents import Document

# 1) 경로
csv_path = "/Users/apple/Team2-RAG-Project/data/raw/data_list.csv"
pre_dir = "/Users/apple/Team2-RAG-Project/data/preprocessed"

# 2) 파일명 키(매칭용) 함수

def file_key(name):
    s = str(name).rsplit(".", 1)[0]
    s = unicodedata.normalize("NFC", s)  # 핵심
    s = re.sub(r"\s+", " ", s).strip()
    return s


# 3) CSV -> {파일키: 메타데이터} 딕셔너리
meta_df = pd.read_csv(csv_path)
meta_df["file_key"] = meta_df["파일명"].apply(file_key)
meta_map = meta_df.set_index("file_key").to_dict("index")

# 4) preprocessed jsonl + CSV 메타 결합 -> Document 생성
documents = []
for fname in sorted(os.listdir(pre_dir)):
    if not fname.endswith(".jsonl"):
        continue

    k = file_key(fname)               # jsonl 파일명 키
    base_meta = meta_map.get(k, {})   # csv 메타데이터

    with open(os.path.join(pre_dir, fname), "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            text = (row.get("text") or "").strip()
            if not text:
                continue

            md = dict(base_meta)
            md["source_file"] = row.get("source_file")
            md["page"] = row.get("page")
            md["author"] = row.get("author")

            documents.append(Document(page_content=text, metadata=md))

print("Document 개수:", len(documents))
print("샘플 metadata:", documents[0].metadata if documents else {})


Document 개수: 7547
샘플 metadata: {'공고 번호': '20240330003', '공고 차수': 0.0, '사업명': '2024년 벤처확인종합관리시스템 기능 고도화 용역사업 입찰공고', '사업 금액': 352000000.0, '발주 기관': '(사)벤처기업협회', '공개 일자': '2024-03-19 10:49:46', '입찰 참여 시작일': nan, '입찰 참여 마감일': '2024-04-09 15:00:00', '사업 요약': '- 2024년 벤처확인종합관리시스템 기능 고도화 용역사업\r\n- 복수의결권주식, 스톡옵션, 성과조건부주식교부 기능 구축 및 고도화\r\n- 벤처기업법에 따라 발행된 복수의결권주식 보고 업무 처리 시스템 구축\r\n- 벤처기업법에 따라 부여된 벤처기업 스톡옵션 부여, 취소-철회 신고 및 업무시스템 구축\r\n- 벤처기업법에 따라 성과조건부주식교부계약의 신고 및 업무처리 시스템 구축\r\n- 복수의결권주식 발행 보고 및 스톡옵션 신고기능은 기존 시스템 내 벤처확인 이력정보와 신고자(벤처기업)의 정보와 연동', '파일형식': 'hwp', '파일명': '(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp', '텍스트': '    \r\n \r\n2024년  벤처확인종합관리시스템 기능 고도화  용역사업\r\n(복수의결권주식, 스톡옵션, 성과조건부주식) -\r\n제안요청서\r\n2024. 03.   \r\n \r\n \r\n   \r\n \r\n목  차\r\n 1. 추진개요\t  -   \t 3\r\n   \r\n 2. 추진방안\t  -   \t 5\r\n   \r\n 3. 추진내용\t  -   \t 9\r\n   \r\n 4. 제안요청내용\t  -   \t 24\r\n   \r\n 5. 입찰관련사항\t  -   \t 78\r\n   \r\n 6. 제안서작성요령\t  -   \t 82\r\n 7. 별지서식 및 붙임\t  -   \t 94\r\n \r\nⅠ. 추진개요\r\n \r\n1\r\n추진

- 전처리 단계에서 공백이었던 29페이지가 삭제되어 (7576 - 29) 총 7547개의 Document 객체가 생성되었다.
- data_list.csv 메타데이터와 결합되었다.

In [34]:
# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(documents)

In [35]:
len(split_documents)

10915

- 총 토큰 수는 7184419개이며,  
 chunk_size=1000에 chunk_overlap=50으로. 
 총 10915개의 chunk가 만들어졌다.

In [36]:
print(split_documents[1].page_content)

목 차
 1. 추진개요·························································································· 3
 2. 추진방안·························································································· 5
 3. 추진내용·························································································· 9
 4. 제안요청내용················································································ 24
 5. 입찰관련사항················································································ 78
 6. 제안서작성요령············································································ 82
 7. 별지서식 및 붙임········································································ 94


In [37]:
# 단계 3: 임베딩(Embedding) 생성
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small", chunk_size=100)
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x138947b50>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x138947ca0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version='', openai_api_base=None, openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=100, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [38]:
# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

In [11]:
# 단계 5: 검색기(Retreiever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성합니다.

import os
import pickle

embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 

DB_PATH  = "/Users/apple/Team2-RAG-Project/data/vectorstore/faiss_openai"
PKL_PATH = "/Users/apple/Team2-RAG-Project/data/vectorstore/split_documents.pkl"

if os.path.exists(DB_PATH) and os.path.exists(PKL_PATH):
    print("기존 DB와 문서를 불러옵니다.")
    vectorstore = FAISS.load_local(DB_PATH, embeddings, allow_dangerous_deserialization=True)
    
    with open(PKL_PATH, "rb") as f:
        split_documents = pickle.load(f)

retriever = vectorstore.as_retriever()
retriever

기존 DB와 문서를 불러옵니다.


VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x135cabbb0>, search_kwargs={})

In [12]:
retriever.invoke("국립중앙의료원의 응급환자 전원 지원 시스템의 핵심 프로세스 중 '상황의사'의 주된 역할은 무엇인가요?")

[Document(metadata={'공고 번호': '20240531542', '공고 차수': 0.0, '사업명': '(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축」 위탁용역', '사업 금액': 1400000000.0, '발주 기관': '국립중앙의료원', '공개 일자': '2024-05-24 11:09:47', '입찰 참여 시작일': '2024-05-27 11:00:00', '입찰 참여 마감일': '2024-06-04 11:00:00', '사업 요약': '- 사업개요: 2024년도에 국립중앙의료원에서 차세대 응급의료 상황관리시스템을 구축하는 사업\r\n- 추진배경: 최근 응급실 섭외 지연으로 인한 문제 발생 및 병원 간 응급환자 전원이 취약한 상황\r\n- 사업범위: 전국 병원 간 전원 및 재난 조정 업무를 위한 콜센터 구축 및 통신장비 도입\r\n- 기대효과: 신속하고 적절한 이송기관 선정, 환자 안전성 향상, 신속한 응급상황 대응, 응급의료 정보 충실도 향상\r\n- 추진목표: 응급환자의 예후 향상, 응급의료 모니터링 및 질 관리, 신속한 응급상황 대응, 응급의료 정보 충실도 향상', '파일형식': 'hwp', '파일명': '국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp', '텍스트': ' \r\n제 안 요 청 서\r\n \r\n사 업 명\r\n2024년도 차세대 응급의료 상황관리시스템 구축\r\n주관기관\r\n국립중앙의료원\r\n2024. 5.\r\n \r\n담당\r\n부 서\r\n성 명\r\n전화번호\r\ne-mail\r\n응급의료정보화팀\r\n이정탁\r\n02-6362-3443\r\nrheejt@nmc.or.kr\r\n \r\n \r\n|\r\n \r\n \r\n \r\n목    차\r\n \r\n \r\n   Ⅰ. 사업개요\r\n  1. 사업안내\t  -   \t\r\n2\r\n  2. 추진배경 및 필요성\t  -   \t\r\n2\r\n  3. 사업범위\t  -   \t\r\n3\r\n  4. 기대효과\t

In [13]:
# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks.
Use the following poeces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Answer in Korean.

#Question:
{question}
#Context:
{context}

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
# 모델(LLM) 를 생성합니다.
llm = ChatOpenAI(model_name="gpt-5-mini", temperature=1)

# 단계 8: 체인(Chain) 생성
langfuse_handler = CallbackHandler() 
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [42]:
# 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "국립중앙의료원의 응급환자 전원 지원 시스템의 핵심 프로세스 중 '상황의사'의 주된 역할은 무엇인가요?"
responce = chain.invoke(question,
                        config={"callbacks": [langfuse_handler]})
print(responce)

문서에 따르면 상황의사의 주된 역할은 다음과 같습니다.

- 접수시스템에 환자정보 입력하고, 필요시 구급대원과 통화해 추가 정보 수집  
- 사전의뢰기관 및 지역별 이송지침을 확인하여 전원(의뢰) 대상 병원 선정 및 수용 요청 진행  
- 수용 가능한 병원을 결정하지 못할 경우 지침에 따라 수용기관을 최종 결정하고, 구급상황관리센터 등으로 통보·연계하여 이송을 확정함


In [43]:
retrieved_chunks = retriever.invoke("국립중앙의료원의 응급환자 전원 지원 시스템의 핵심 프로세스 중 '상황의사'의 주된 역할은 무엇인가요?")
for i, doc in enumerate(retrieved_chunks):
    page_num = doc.metadata.get("page", "정보 없음")
    print(page_num) 

7
1
3
2


In [14]:
question = "나노종합기술원의 스마트 팹 설비온라인 시스템 구축 용역은 계약 후 기간이 총 얼마나 소요되나요?"
responce = chain.invoke(question,
                        config={"callbacks": [langfuse_handler]})
print(responce)

계약체결일로부터 18개월입니다.


In [45]:
retrieved_chunks = retriever.invoke("나노종합기술원의 스마트 팹 설비온라인 시스템 구축 용역은 계약 후 기간이 총 얼마나 소요되나요?")
for i, doc in enumerate(retrieved_chunks):
    page_num = doc.metadata.get("page", "정보 없음")
    print(page_num) 

0
2
27
25


#### 평가 데이터셋
- Q1. "국립중앙의료원의 응급환자 전원 지원 시스템의 핵심 프로세스 중 '상황의사'의 주된 역할은 무엇인가요?"

- A."의뢰 대상 기관 선정, 수용 의뢰 및 상황 지원을 담당합니다."

- 14페이지 [업무 흐름도] 설명부, 이미지 정보

- Q2. "나노종합기술원의 스마트 팹 설비온라인 시스템 구축 용역은 계약 후 기간이 총 얼마나 소요되나요?"

- A."총 18개월입니다."

- "1페이지 15째 줄"

#### 결과:
- Q1: '수용의뢰'는 맞췄으나, 상황 지원은 맞추지 못함
- Q2: 18개월을 정확히 맞추었다.

In [15]:
# 평가 코드 import
import sys
from pathlib import Path

PROJECT_ROOT = Path('/Users/apple/Team2-RAG-Project')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from evaluation.evaluate_final import run_evaluation


In [16]:
# 평가 설정
DATASET_PATH = "/Users/apple/Team2-RAG-Project/evaluation/test_questions.json"
THRESHOLD = 0.35


In [17]:
results = run_evaluation(
    chain=chain,
    dataset_path=DATASET_PATH,
    threshold=THRESHOLD,
    use_langfuse=False,
)


RAG 시스템 평가 시작...

================= DEBUG Q01 =================
Q: 봉화군에서 진행하는 재난 시스템 고도화 사업 예산이 총 얼마로 잡혀 있나요? 주관 부서가 어디인지도 확인 부탁드립니다.
GOLD: 900,000,000원(VAT 포함)이며, 봉화군 안전건설과에서 주관합니다.

--- PRED_CORE(300) ---
총 사업비는 900,000,000원(부가세 포함, 약 9억원)이며, 주관 부서는 경상북도 봉화군(봉화군청) 안전건설과입니다.

Q01: [O]
================= DEBUG Q02 =================
Q: 이번 봉화군 재난위험지역 경보발령 사업에서 활용하는 무선 통신 방식은 구체적으로 무엇인가요?
GOLD: LTE 및 DMB(EWS) 방식을 활용합니다.

--- PRED_CORE(300) ---
문서에 따르면 무선 통신 방식은 LTE와 DMB(EWS)입니다.  
(즉 디지털 데이터방송기술(DMB EWS) 및 LTE를 통한 무선 방식의 경보방송을 활용합니다.)

Q02: [O]
================= DEBUG Q03 =================
Q: 봉화군 재난 사업용 하드웨어 규격 중에서 ECR-02 무선경보장치 스피커의 출력 기준을 알려주세요.
GOLD: 100W 이상의 출력을 지원해야 합니다.

--- PRED_CORE(300) ---
ECR-02 규격에서 스피커(앰프) 출력 기준은 총 앰프 출력 400W 이상(100W × 4채널)입니다.

Q03: [O]
================= DEBUG Q04 =================
Q: 고려대 차세대 학사시스템 구축 사업의 전체 예산 규모와 연도별 분할 지급 비율이 어떻게 되나요?
GOLD: 총 11,270,000,000원이며, 2024년 30%, 2025년 40%, 2026년 30%를 지급합니다.

--- PRED_CORE(300) ---
고려대학교 차세대 포털·학사 정보시스템 구축 사업의 예산은

| 유형 | 문항 번호 (ID) | 문항 수 | 주요 평가 요소 |
|---|---|---:|---|
| 텍스트 | 5, 6, 16, 19, 22, 25, 26, 27, 28 | 9 | 문맥 파악 및 비정형 데이터 추출 |
| 표 | 1, 3, 4, 8, 10, 12, 14, 17, 20, 24, 29, 30 | 12 | 표 구조 분석 및 정확한 수치 매칭 |
| 이미지 | 2, 7, 9, 11, 13, 15, 18, 21, 23 | 9 | 시각 도식 해석 및 논리적 흐름 파악 |

정답: 1, 2, 4, 7, 10, 13
- 텍스트보다 이미지, 표에 있는 정보가 잘 추출된 것을 확인할 수 있다.
텍스트 형식으로 추출된 표에 정보가 잘 저장되있다거나, 이미지가 추출되지 않더라도 텍스트 형식으로도 문서에 있는 것으로 추정할 수 있다.